<a href="https://colab.research.google.com/github/nehasmhs/Financial-AI-Hackathon/blob/main/Annual_Financial_Statement_Generali_GroupBank_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Financial Bank Performance Analyzer

Automated analysis pipeline for Generali Group Bank Annual Reports using:
- **LandingAI's ADE** for document extraction
- **AWS S3** for data storage
- **AWS Bedrock Knowledge Base** for semantic search
- **Strands Agent Framework** with Claude Sonnet 4.0

In [2]:
!pip install boto3 python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.4/139.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 104.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 6.1 MB/s eta 0:00:00


In [11]:
!pip install strands

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for strands (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for strands
Failed to build strands
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (strands)


In [14]:
!pip install strands-agents strands-agents-tools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.7/249.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.4/243.4 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.1/230.1 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 1. Setup and Configuration

In [6]:
import os
import json
import boto3
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

LANDINGAI_API_KEY = os.getenv('LANDINGAI_API_KEY')
AWS_REGION = os.getenv('AWS_REGION', 'eu-central-1')
S3_BUCKET_NAME = 'financial-reports-generali-bank'
BEDROCK_DATA_SOURCE_ID = os.getenv('BEDROCK_DATA_SOURCE_ID')
BEDROCK_MODEL_ID = os.getenv('BEDROCK_MODEL_ID', 'us.anthropic.claude-sonnet-4-5-20250929-v1:0')
BEDROCK_KB_ID = os.getenv('BEDROCK_KB_ID')

print("✓ Configuration loaded")
print(f"  - AWS Region: {AWS_REGION}")
print(f"  - S3 Bucket: {S3_BUCKET_NAME}")
print(f"  - Model: {BEDROCK_MODEL_ID}")

✓ Configuration loaded
  - AWS Region: eu-central-1
  - S3 Bucket: financial-reports-generali-bank
  - Model: us.anthropic.claude-sonnet-4-5-20250929-v1:0


## 2. Load Extracted Document Data

In [4]:
# Load from markdown or JSON file
markdown_file = 'Annual Financial Statements 2024_Generali Group.extraction.md'
json_file = 'Annual Financial Statements 2024_Generali Group.extraction.json'

if os.path.exists(markdown_file):
    print(f"Loading from: {markdown_file}")
    with open(markdown_file, 'r') as f:
        markdown_content = f.read()
    extracted_data = {
        'result': {
            'markdown': markdown_content
        }
    }
    print(f"✓ Loaded {len(markdown_content)} characters")
elif os.path.exists(json_file):
    print(f"Loading from: {json_file}")
    with open(json_file, 'r') as f:
        extracted_data = json.load(f)
    print(f"✓ Loaded extraction result")
else:
    raise FileNotFoundError(f"Please place '{markdown_file}' or '{json_file}' in this directory")

Loading from: Annual Financial Statements 2024_Generali Group.extraction.md
✓ Loaded 336274 characters


## 3. Upload to S3

In [12]:
s3_client = boto3.client('s3', region_name=AWS_REGION)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
s3_key = f"financial-reports-generali-bank_{timestamp}.json"

s3_client.put_object(
    Bucket=S3_BUCKET_NAME,
    Key=s3_key,
    Body=json.dumps(extracted_data, indent=2),
    ContentType='application/json'
)

s3_uri = f"s3://{S3_BUCKET_NAME}/{s3_key}"
print(f"✓ Data uploaded to: {s3_uri}")

✓ Data uploaded to: s3://financial-reports-generali-bank/financial-reports-generali-bank_20251110_123644.json


In [8]:
import pandas as pd
import os

# Load the access keys from the CSV file
try:
    credentials_df = pd.read_csv('neha2_accessKeys.csv')

    # Extract credentials
    aws_access_key_id = credentials_df.loc[0, 'Access key ID']
    aws_secret_access_key = credentials_df.loc[0, 'Secret access key']

    # Set environment variables
    os.environ['AWS_ACCESS_KEY_ID'] = aws_access_key_id
    os.environ['AWS_SECRET_ACCESS_KEY'] = aws_secret_access_key

    print("✓ AWS credentials loaded from neha2_accessKeys.csv and set as environment variables.")
except FileNotFoundError:
    print("Error: neha2_accessKeys.csv not found. Please ensure it's in the correct directory.")
except KeyError as e:
    print(f"Error: Missing expected column in neha2_accessKeys.csv: {e}. Please check the CSV format.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

✓ AWS credentials loaded from neha2_accessKeys.csv and set as environment variables.


## 4. Sync Bedrock Knowledge Base

In [9]:
if BEDROCK_KB_ID:
    bedrock_agent = boto3.client('bedrock-agent', region_name=AWS_REGION)

    response = bedrock_agent.start_ingestion_job(
        knowledgeBaseId=BEDROCK_KB_ID,
        dataSourceId=BEDROCK_DATA_SOURCE_ID
    )

    print(f"✓ Knowledge base sync initiated")
    print(f"  - Job ID: {response.get('ingestionJob', {}).get('ingestionJobId')}")
else:
    print("⚠ BEDROCK_KB_ID not set - skipping KB sync")

⚠ BEDROCK_KB_ID not set - skipping KB sync


## 5. Create Financial Performance Analyst Tools

In [16]:
import strands

@strands.tool
def search_generali_report(query: str, max_results: int = 5) -> dict:
    """
    Search Generali Group's 2024 Annual Integrated Report and Consolidated Financial Statements for banking and insurance insights.

    Use this tool to find:
    - Interest rate trends and impacts on Life and PC segments
    - Market conditions and analysis by country and segment
    - Policy impacts and regulatory changes (e.g. IFRS, Solvency)
    - Financial metrics/statistics: premiums, net inflows, operating results, ratios
    - Historical comparisons, segmental trends, country breakouts

    Args:
        query: Natural language question or search query (e.g. "Italy interest rates", "Life segment operating result")
        max_results: Maximum number of results to return (default: 5)

    Returns:
        Highlighted passages from the Generali Group report with context (country, segment, year)
    """
    # Check if the PDF path is set, or load as needed
    PDF_PATH = "Annual-Integrated-Report-and-Consolidated-Financial-Statements-2024_Generali-Group_final_interactive-pages.pdf"

    try:
        from some_pdf_search_lib import search_pdf  # pseudocode placeholder; use your own search utility

        # Perform keyword search or semantic retrieval
        results = search_pdf(
            file_path=PDF_PATH,
            query=query,
            max_results=max_results,
            context_window=350  # chars before/after
        )

        return {
            'query': query,
            'results_count': len(results),
            'results': [
                {
                    'passage': r['snippet'],
                    'page': r.get('page_num', None),
                    'score': r.get('score', None),
                    'context': r.get('context', None)
                }
                for r in results
            ]
        }

    except Exception as e:
        return {
            'error': f'Report search failed: {str(e)}',
            'query': query
        }

print("✓ Generali Group report search tool created")


✓ Generali Group report search tool created


Below one to be deleted

In [15]:
import strands

@strands.tool
def search_knowledge_base(query: str, max_results: int = 5) -> dict:
    """
    Search the Bedrock Knowledge Base for relevant information about Generali Group banking.

    Use this tool to find:
    - Interest rate trends and data
    - Market conditions and analysis
    - Policy impacts and regulatory information
    - Financial metrics and statistics
    - Historical comparisons and trends

        Use this tool to find:
    - Interest rate trends and impacts on Life and PC segments
    - Market conditions and analysis by country and segment
    - Policy impacts and regulatory changes (e.g. IFRS, Solvency)
    - Financial metrics/statistics: premiums, net inflows, operating results, ratios
    - Historical comparisons, segmental trends, country breakouts

    Args:
        query: Natural language question or search query
        max_results: Maximum number of results to return (default: 5)

    Returns:
        Relevant passages from the Federal Reserve report with context
    """
    if not BEDROCK_KB_ID:
        return {'error': 'Knowledge base not configured. Please set BEDROCK_KB_ID in .env file.'}

    try:
        bedrock_agent_runtime = boto3.client('bedrock-agent-runtime', region_name=AWS_REGION)

        response = bedrock_agent_runtime.retrieve(
            knowledgeBaseId=BEDROCK_KB_ID,
            retrievalQuery={'text': query},
            retrievalConfiguration={
                'vectorSearchConfiguration': {
                    'numberOfResults': max_results
                }
            }
        )

        results = []
        for item in response.get('retrievalResults', []):
            results.append({
                'content': item.get('content', {}).get('text', ''),
                'score': item.get('score', 0),
                'location': item.get('location', {})
            })

        return {
            'query': query,
            'results_count': len(results),
            'results': results
        }

    except Exception as e:
        return {
            'error': f'Knowledge base search failed: {str(e)}',
            'query': query
        }

print("✓ Knowledge Base search tool created")

✓ Knowledge Base search tool created


## 6. Initialize Strands Agent

In [17]:
from strands import Agent

agent = Agent(
    model=BEDROCK_MODEL_ID,
    name="Financial Bank Performance Analyzer",
    description="Expert agent for analyzing Generali Group banking annual report",
    system_prompt="""
You are a Financial Bank Performance Analyzer specializing in credit card banking and Federal Reserve reports.

When responding:
- If a question is ambiguous, ask clarifying questions before providing analysis
- If you need specific parameters (time periods, metrics, segments), ask the user to specify
- Use your tools to gather data, then provide clear, data-driven insights
- Be conversational and interactive

You have access to structured data extracted from Federal Reserve credit card banking reports.
Provide clear, data-driven insights backed by the available financial data.
""",
    tools=[search_knowledge_base]
)

print("✓ Agent initialized")
print(f"  - Model: {BEDROCK_MODEL_ID}")
print(f"  - Tools: {len(agent.tool_names)}")

✓ Agent initialized
  - Model: us.anthropic.claude-sonnet-4-5-20250929-v1:0
  - Tools: 1


## 7. Interactive Chat Loop

In [18]:
print("="*70)
print("Financial Bank Performance Analyzer - Interactive Chat")
print("="*70)
print("\nAsk questions about the Generali Group Bank report.")
print("Type 'exit', 'quit', or 'bye' to end the conversation.")
print("="*70 + "\n")

while True:
    try:
        user_input = input("\n🧑 You: ").strip()

        if not user_input:
            continue

        if user_input.lower() in ['exit', 'quit', 'bye', 'q']:
            print("\n👋 Ending conversation. Goodbye!")
            break

        print("\n🤖 Agent: ", end="")
        result = agent(user_input)
        print(result)

    except KeyboardInterrupt:
        print("\n\n👋 Conversation interrupted. Goodbye!")
        break
    except Exception as e:
        print(f"\n❌ Error: {e}")
        print("Please try again or type 'exit' to quit.")

Financial Bank Performance Analyzer - Interactive Chat

Ask questions about the Generali Group Bank report.
Type 'exit', 'quit', or 'bye' to end the conversation.


🧑 You: Hi 

🤖 Agent: 
❌ Error: An error occurred (ValidationException) when calling the ConverseStream operation: Operation not allowed
Please try again or type 'exit' to quit.


👋 Conversation interrupted. Goodbye!

🧑 You: Tell me about the bank
